# Summary_Day8.ipynb  
## 이진 분류 · Sigmoid · BCE · BCEWithLogitsLoss · 평가 지표 · 불균형 데이터 · 실무형 분류 파이프라인

이번 8강은 **이진 분류 Binary Classification**를 정리하는 강의다.

앞에서 배운 회귀는 연속적인 숫자를 예측하는 문제였다.  
이번 강의의 이진 분류는 데이터를 두 그룹 중 하나로 나누는 문제다.

```text
회귀: 집값이 얼마인가?
이진 분류: 사기 거래인가 아닌가?
```

이번 강의의 큰 흐름은 다음이다.

```text
이진 분류 문제 정의
→ Sigmoid로 확률 만들기
→ 0.5 threshold로 class 예측
→ BCE / BCEWithLogitsLoss로 손실 계산
→ Accuracy로 기본 성능 확인
→ Precision, Recall, F1, ROC-AUC로 더 자세히 평가
→ Confusion Matrix로 예측 결과 구조 확인
→ 클래스 불균형에서 pos_weight 사용
→ 사기 거래 탐지, 당뇨병 예측, 고객 이탈 예측으로 실무형 흐름 확장
```

> 필기 포인트:  
> 이진 분류에서 모델 출력은 보통 “1일 확률”로 해석한다.  
> 하지만 학습할 때는 `BCEWithLogitsLoss`를 쓰면 Sigmoid 전 raw score인 logit을 그대로 넣는 것이 더 안정적이다.

## 1. 라이브러리 준비

이번 실습에서는 PyTorch, NumPy, pandas, Matplotlib, scikit-learn을 사용한다.

### 함수/모듈 사용법

```python
import torch
import torch.nn as nn
import torch.optim as optim
```

- `torch`: Tensor와 자동 미분을 사용할 때 사용한다.
- `nn`: 모델 Layer와 손실함수를 만들 때 사용한다.
- `optim`: Optimizer를 만들 때 사용한다.
- `train_test_split`: 데이터를 train/test로 나눌 때 사용한다.
- `StandardScaler`: 수치형 feature를 표준화할 때 사용한다.
- `confusion_matrix`: 혼동행렬을 계산한다.
- `classification_report`: precision, recall, f1-score를 한 번에 출력한다.
- `roc_auc_score`: ROC-AUC 값을 계산한다.

> 실습 메모:  
> 원본 강의에는 Iris 이진 분류, 사기 거래 탐지, 당뇨병 예측, 고객 이탈 예측 예제가 있다.  
> 이 노트북은 인터넷 없이 실행되도록 scikit-learn 내장 데이터와 합성 데이터를 사용한다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_iris, make_classification, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

import joblib

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("PyTorch:", torch.__version__)

## 2. 이진 분류란 무엇인가

이진 분류는 데이터를 두 그룹 중 하나로 나누는 작업이다.

예시는 다음과 같다.

```text
스팸 메일 / 정상 메일
사기 거래 / 정상 거래
당뇨병 고위험 / 저위험
고객 이탈 / 고객 유지
```

회귀와의 차이는 다음이다.

| 문제 | 출력 |
|---|---|
| 회귀 | 연속적인 숫자 |
| 이진 분류 | 0 또는 1 |

> 기억할 점:  
> 이진 분류는 “정답이 1일 확률”을 만든 뒤, 임계값 threshold를 기준으로 0 또는 1을 결정한다.

In [ ]:
examples = {
    "spam": "스팸 메일이면 1, 정상 메일이면 0",
    "fraud": "사기 거래이면 1, 정상 거래이면 0",
    "diabetes": "고위험이면 1, 저위험이면 0",
    "churn": "이탈이면 1, 유지이면 0",
}

for key, value in examples.items():
    print(f"{key}: {value}")

## 3. Sigmoid 함수

Sigmoid 함수는 어떤 숫자가 들어와도 0과 1 사이 값으로 바꾼다.

공식은 다음이다.

```text
sigmoid(x) = 1 / (1 + exp(-x))
```

이진 분류에서는 Sigmoid 출력값을 “class 1일 확률”처럼 해석한다.

### 함수 사용법

```python
torch.sigmoid(x)
```

- `x`: logit, 즉 Sigmoid 전 raw score다.
- 반환값은 0과 1 사이의 확률값이다.

In [ ]:
x_sigmoid = torch.linspace(-6, 6, 121)
y_sigmoid = torch.sigmoid(x_sigmoid)

print("x=0일 때 sigmoid:", torch.sigmoid(torch.tensor(0.0)).item())

In [ ]:
plt.plot(x_sigmoid.numpy(), y_sigmoid.numpy())
plt.axhline(0.5, linestyle="--")
plt.axvline(0.0, linestyle="--")
plt.xlabel("logit")
plt.ylabel("sigmoid(logit)")
plt.title("Sigmoid Function")
plt.show()

그래프 해석:

- 입력이 0이면 출력은 0.5다.
- 입력이 커질수록 1에 가까워진다.
- 입력이 작아질수록 0에 가까워진다.
- 그래서 0.5를 기준으로 0/1 class를 나눌 수 있다.

## 4. Logit, Probability, Threshold

이진 분류에서 자주 헷갈리는 세 단어다.

| 이름 | 의미 |
|---|---|
| logit | Sigmoid 전 모델 raw output |
| probability | Sigmoid를 지난 0~1 확률 |
| threshold | 확률을 class로 바꾸는 기준값 |

기본 기준은 보통 0.5다.

```text
probability >= 0.5 → class 1
probability < 0.5 → class 0
```

In [ ]:
logits = torch.tensor([-2.0, -0.2, 0.0, 0.7, 2.0])
probs = torch.sigmoid(logits)
preds = (probs >= 0.5).float()

print("logits:", logits)
print("probs:", probs)
print("preds:", preds)

> 필기 포인트:  
> `BCEWithLogitsLoss`를 사용할 때는 모델 출력 logits를 손실함수에 바로 넣는다.  
> 예측 확률이 필요할 때만 `torch.sigmoid(logits)`를 적용한다.

## 5. BCE와 Cross Entropy 직관

이진 분류에서는 회귀의 MSE보다 BCE 계열 손실함수를 사용한다.

BCE는 Binary Cross Entropy다.

핵심 직관은 다음이다.

```text
정답을 맞혔고 확신도 높음 → 작은 loss
정답을 틀렸지만 애매하게 틀림 → 중간 loss
정답을 틀렸고 강하게 확신함 → 큰 loss
```

> 강의 비유:  
> “정답은 사과인 것 같아요”라고 틀린 학생보다,  
> “정답은 100% 바나나이다”라고 틀린 학생에게 더 큰 벌점을 주는 방식이다.

In [ ]:
targets = torch.tensor([1.0, 1.0, 1.0])
probs_good = torch.tensor([0.9, 0.8, 0.7])
probs_bad = torch.tensor([0.1, 0.2, 0.3])

bce = nn.BCELoss()

print("정답 방향 예측 loss:", bce(probs_good, targets).item())
print("반대 방향 예측 loss:", bce(probs_bad, targets).item())

## 6. BCELoss 사용법

`BCELoss`는 Sigmoid를 지난 확률값을 입력으로 받는다.

### 함수 사용법

```python
criterion = nn.BCELoss()
loss = criterion(prob, target)
```

- `prob`: 0과 1 사이 확률값이다.
- `target`: 0 또는 1 정답이다.
- 모델 마지막에 Sigmoid가 있어야 한다.

> 주의:  
> raw logit을 `BCELoss`에 바로 넣으면 안 된다.

In [ ]:
prob = torch.tensor([[0.9], [0.2], [0.7], [0.1]])
target = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

criterion_bce = nn.BCELoss()
loss_bce = criterion_bce(prob, target)

print("BCELoss:", loss_bce.item())

## 7. BCEWithLogitsLoss 사용법

실무에서는 `BCEWithLogitsLoss`를 더 권장한다.

### 함수 사용법

```python
criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, target)
```

- `logits`: Sigmoid 전 raw score다.
- `target`: 0 또는 1 정답이다.
- 내부에서 Sigmoid와 BCE를 안정적으로 함께 처리한다.

> 시험 포인트:  
> `BCEWithLogitsLoss`를 쓸 때 모델 마지막에 Sigmoid를 붙이지 않는다.

In [ ]:
logits = torch.tensor([[2.2], [-1.4], [1.1], [-2.0]])
target = torch.tensor([[1.0], [0.0], [1.0], [0.0]])

criterion_logits = nn.BCEWithLogitsLoss()
loss_logits = criterion_logits(logits, target)

manual_prob = torch.sigmoid(logits)
manual_loss = nn.BCELoss()(manual_prob, target)

print("BCEWithLogitsLoss:", loss_logits.item())
print("Sigmoid + BCELoss:", manual_loss.item())

정리:

```text
BCELoss 방식:
Linear → Sigmoid → BCELoss
예측 threshold: probability 0.5

BCEWithLogitsLoss 방식:
Linear → BCEWithLogitsLoss
예측 threshold: logit 0 또는 probability 0.5
```

BCEWithLogitsLoss가 수치적으로 더 안정적이다.

## 8. Iris 데이터 준비

강의의 기본 예제는 Iris 데이터셋이다.

원본 Iris는 3개 품종과 4개 feature를 가진다.  
이번 이진 분류에서는 앞의 두 품종 100개와 앞의 두 feature만 사용한다.

```text
입력 X: sepal length, sepal width
정답 y: class 0 또는 class 1
```

### 함수 사용법: `load_iris()`

```python
iris = load_iris()
```

- scikit-learn 내장 Iris 데이터를 불러온다.
- `iris.data`: feature 배열이다.
- `iris.target`: class label 배열이다.

In [ ]:
iris = load_iris()

x_data = iris.data[:100, :2]
y_data = iris.target[:100]

print("iris.data shape:", iris.data.shape)
print("iris.target shape:", iris.target.shape)
print("x_data shape:", x_data.shape)
print("y_data shape:", y_data.shape)
print("class counts:", np.unique(y_data, return_counts=True))

## 9. Train / Validation 분할

데이터를 학습용과 검증용으로 나눈다.

### 함수 사용법

```python
train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
```

- `test_size=0.3`: 전체의 30%를 검증 데이터로 둔다.
- `stratify=y`: class 비율이 train/validation에 비슷하게 유지되도록 한다.
- `random_state=42`: 랜덤 분할 결과를 고정한다.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_data,
    y_data,
    test_size=0.3,
    random_state=42,
    stratify=y_data
)

print("x_train:", x_train.shape)
print("x_val:", x_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("train class counts:", np.unique(y_train, return_counts=True))
print("val class counts:", np.unique(y_val, return_counts=True))

> 기억할 점:  
> 학습 데이터는 모델이 공부하는 연습 문제다.  
> 검증 데이터는 처음 보는 모의고사 역할이다.

## 10. Iris 산점도 확인

두 class가 feature 공간에서 어떻게 나뉘는지 먼저 눈으로 본다.

In [ ]:
x0 = x_train[y_train == 0]
x1 = x_train[y_train == 1]

plt.scatter(x0[:, 0], x0[:, 1], marker="x", label="class 0")
plt.scatter(x1[:, 0], x1[:, 1], marker="o", label="class 1")
plt.xlabel("sepal length")
plt.ylabel("sepal width")
plt.title("Iris Binary Classification Train Data")
plt.legend()
plt.show()

그래프 해석:

- 점 하나가 붓꽃 하나다.
- 두 class가 어느 정도 직선으로 나뉠 수 있는 형태다.
- 로지스틱 회귀는 이 공간을 가르는 결정 경계선을 학습한다.

## 11. Tensor 변환

PyTorch 모델 학습을 위해 NumPy 배열을 Tensor로 바꾼다.

### 함수 사용법

```python
torch.tensor(x_train).float()
torch.tensor(y_train).float().view(-1, 1)
```

- 입력 feature는 float Tensor로 만든다.
- 정답도 `BCE` 계열 손실에 맞춰 float Tensor로 만든다.
- `view(-1, 1)`로 `[N]`을 `[N, 1]` 형태로 맞춘다.

In [ ]:
inputs = torch.tensor(x_train).float()
labels = torch.tensor(y_train).float().view(-1, 1)

inputs_val = torch.tensor(x_val).float()
labels_val = torch.tensor(y_val).float().view(-1, 1)

print("inputs:", inputs.shape)
print("labels:", labels.shape)
print("inputs_val:", inputs_val.shape)
print("labels_val:", labels_val.shape)

## 12. Sigmoid를 포함한 로지스틱 회귀 모델

첫 번째 모델은 `Linear → Sigmoid` 구조다.

```text
입력 2개 → Linear → 출력 1개 → Sigmoid → 확률
```

### 모델 사용법

```python
net = LogisticNetWithSigmoid(n_input=2, n_output=1)
outputs = net(inputs)
```

- 출력은 이미 0과 1 사이 확률값이다.
- 손실함수는 `nn.BCELoss()`를 사용한다.

In [ ]:
class LogisticNetWithSigmoid(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)
        self.sigmoid = nn.Sigmoid()

        self.l1.weight.data.fill_(1.0)
        self.l1.bias.data.fill_(1.0)

    def forward(self, x):
        x1 = self.l1(x)
        x2 = self.sigmoid(x1)
        return x2

net_bce = LogisticNetWithSigmoid(2, 1)

print(net_bce)

> 헷갈림 포인트:  
> 모델 안에 Sigmoid가 있으면 출력은 확률값이다.  
> 그래서 `BCELoss`를 쓴다.

## 13. BCELoss 방식 학습 루프

BCELoss 방식 학습 순서는 다음이다.

```text
optimizer.zero_grad()
→ outputs = net(inputs)
→ loss = BCELoss(outputs, labels)
→ loss.backward()
→ optimizer.step()
```

정확도는 확률값을 0.5 기준으로 나눠 계산한다.

In [ ]:
def accuracy_from_prob(prob, labels):
    pred = (prob >= 0.5).float()
    acc = (pred == labels).float().mean()
    return acc.item()

criterion_bce = nn.BCELoss()
optimizer_bce = optim.SGD(net_bce.parameters(), lr=0.01)

num_epochs = 3000
history_bce = []

for epoch in range(num_epochs + 1):
    net_bce.train()
    optimizer_bce.zero_grad()

    outputs = net_bce(inputs)
    loss = criterion_bce(outputs, labels)

    loss.backward()
    optimizer_bce.step()

    if epoch % 100 == 0:
        net_bce.eval()
        with torch.no_grad():
            train_prob = net_bce(inputs)
            val_prob = net_bce(inputs_val)

            train_loss = criterion_bce(train_prob, labels).item()
            val_loss = criterion_bce(val_prob, labels_val).item()

            train_acc = accuracy_from_prob(train_prob, labels)
            val_acc = accuracy_from_prob(val_prob, labels_val)

        history_bce.append([epoch, train_loss, val_loss, train_acc, val_acc])

history_bce = np.array(history_bce)

print("초기 val acc:", history_bce[0, 4])
print("최종 val acc:", history_bce[-1, 4])

### 함수 사용법: `accuracy_from_prob()`

```python
accuracy_from_prob(prob, labels)
```

- `prob >= 0.5`로 class를 예측한다.
- 예측값과 정답을 비교한다.
- 맞은 비율을 반환한다.

In [ ]:
plt.plot(history_bce[:, 0], history_bce[:, 1], label="train loss")
plt.plot(history_bce[:, 0], history_bce[:, 2], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("BCELoss Training Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history_bce[:, 0], history_bce[:, 3], label="train acc")
plt.plot(history_bce[:, 0], history_bce[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("BCELoss Accuracy")
plt.legend()
plt.show()

그래프 해석:

- loss가 내려가면 모델이 정답에 가까워지고 있다는 뜻이다.
- accuracy가 올라가면 실제 class를 더 많이 맞힌다는 뜻이다.
- train과 validation이 비슷하게 움직이면 과적합이 심하지 않다고 볼 수 있다.

## 14. BCEWithLogitsLoss 방식 모델

권장 방식은 모델 마지막에 Sigmoid를 넣지 않는 것이다.

```text
입력 2개 → Linear → logit 1개
```

손실함수는 `BCEWithLogitsLoss`를 사용한다.

확률이 필요할 때만 평가 단계에서 `torch.sigmoid(logits)`를 적용한다.

In [ ]:
class LogisticNetLogits(nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()
        self.l1 = nn.Linear(n_input, n_output)

        self.l1.weight.data.fill_(1.0)
        self.l1.bias.data.fill_(1.0)

    def forward(self, x):
        return self.l1(x)

net_logits = LogisticNetLogits(2, 1)

print(net_logits)

> 기억할 점:  
> `BCEWithLogitsLoss` 내부에 Sigmoid가 들어있다고 생각하면 된다.

## 15. BCEWithLogitsLoss 방식 학습 루프

학습 중에는 logits를 손실함수에 바로 넣는다.  
정확도를 계산할 때만 Sigmoid를 적용한다.

In [ ]:
def accuracy_from_logits(logits, labels):
    prob = torch.sigmoid(logits)
    pred = (prob >= 0.5).float()
    acc = (pred == labels).float().mean()
    return acc.item()

criterion_logits = nn.BCEWithLogitsLoss()
optimizer_logits = optim.SGD(net_logits.parameters(), lr=0.01)

history_logits = []

for epoch in range(num_epochs + 1):
    net_logits.train()
    optimizer_logits.zero_grad()

    logits_train = net_logits(inputs)
    loss = criterion_logits(logits_train, labels)

    loss.backward()
    optimizer_logits.step()

    if epoch % 100 == 0:
        net_logits.eval()
        with torch.no_grad():
            train_logits = net_logits(inputs)
            val_logits = net_logits(inputs_val)

            train_loss = criterion_logits(train_logits, labels).item()
            val_loss = criterion_logits(val_logits, labels_val).item()

            train_acc = accuracy_from_logits(train_logits, labels)
            val_acc = accuracy_from_logits(val_logits, labels_val)

        history_logits.append([epoch, train_loss, val_loss, train_acc, val_acc])

history_logits = np.array(history_logits)

print("초기 val acc:", history_logits[0, 4])
print("최종 val acc:", history_logits[-1, 4])

In [ ]:
plt.plot(history_logits[:, 0], history_logits[:, 1], label="train loss")
plt.plot(history_logits[:, 0], history_logits[:, 2], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("BCEWithLogitsLoss Training Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history_logits[:, 0], history_logits[:, 3], label="train acc")
plt.plot(history_logits[:, 0], history_logits[:, 4], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("BCEWithLogitsLoss Accuracy")
plt.legend()
plt.show()

## 16. 결정 경계 Decision Boundary

로지스틱 회귀 모델은 feature 공간을 가르는 하나의 선을 학습한다.

이 선을 결정 경계라고 한다.

BCEWithLogitsLoss 방식에서는 class 기준을 다음처럼 볼 수 있다.

```text
logit >= 0 → class 1
logit < 0 → class 0
```

선형 모델이므로 결정 경계도 직선이다.

In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()]).float()

    model.eval()
    with torch.no_grad():
        out = model(grid)

        if out.min() >= 0 and out.max() <= 1:
            prob = out.numpy().reshape(xx.shape)
        else:
            prob = torch.sigmoid(out).numpy().reshape(xx.shape)

    plt.contourf(xx, yy, prob, levels=20, alpha=0.4)
    plt.contour(xx, yy, prob, levels=[0.5], linewidths=2)

    plt.scatter(X[y == 0, 0], X[y == 0, 1], marker="x", label="class 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], marker="o", label="class 1")

    plt.xlabel("sepal length")
    plt.ylabel("sepal width")
    plt.title(title)
    plt.legend()
    plt.show()

plot_decision_boundary(net_logits, x_val, y_val, "Decision Boundary on Validation Data")

그래프 해석:

- 색 배경은 class 1일 확률이다.
- 가운데 선은 확률 0.5인 지점이다.
- 이 선을 기준으로 두 class가 나뉜다.

## 17. 기본 평가 지표: Accuracy

Accuracy는 전체 데이터 중 몇 개를 맞혔는지 나타낸다.

```text
Accuracy = 맞은 개수 / 전체 개수
```

### 함수 사용법

```python
accuracy_score(y_true, y_pred)
```

- `y_true`: 실제 정답이다.
- `y_pred`: 예측 class다.

In [ ]:
net_logits.eval()

with torch.no_grad():
    val_logits = net_logits(inputs_val)
    val_prob = torch.sigmoid(val_logits)
    val_pred = (val_prob >= 0.5).int().numpy().ravel()

y_val_np = y_val.astype(int)

acc = accuracy_score(y_val_np, val_pred)

print("Validation Accuracy:", acc)

> 주의:  
> Accuracy는 직관적이지만 클래스 불균형이 있으면 위험할 수 있다.  
> 사기 거래처럼 양성 class가 매우 적은 문제에서는 precision, recall, f1도 봐야 한다.

## 18. Precision, Recall, F1

이진 분류에서는 정확도만 보면 부족할 수 있다.

| 지표 | 의미 |
|---|---|
| Precision | 1이라고 예측한 것 중 진짜 1의 비율 |
| Recall | 실제 1 중에서 모델이 찾아낸 비율 |
| F1 | Precision과 Recall의 균형 |

In [ ]:
prec = precision_score(y_val_np, val_pred, zero_division=0)
rec = recall_score(y_val_np, val_pred, zero_division=0)
f1 = f1_score(y_val_np, val_pred, zero_division=0)

print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)

> 필기 포인트:  
> 사기 탐지나 질병 예측에서는 Recall이 특히 중요할 수 있다.  
> 놓치면 안 되는 양성 class를 얼마나 잘 찾는지가 중요하기 때문이다.

## 19. Confusion Matrix

Confusion Matrix는 예측 결과를 2×2 표로 보여준다.

```text
TN: 실제 0, 예측 0
FP: 실제 0, 예측 1
FN: 실제 1, 예측 0
TP: 실제 1, 예측 1
```

### 함수 사용법

```python
confusion_matrix(y_true, y_pred)
```

In [ ]:
cm = confusion_matrix(y_val_np, val_pred)

print(cm)

tn, fp, fn, tp = cm.ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

In [ ]:
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm):
    plt.text(j, i, str(value), ha="center", va="center")

plt.colorbar()
plt.show()

그래프 해석:

- 대각선 값이 맞힌 개수다.
- 오른쪽 위 FP는 실제 0인데 1이라고 예측한 경우다.
- 왼쪽 아래 FN은 실제 1인데 0이라고 놓친 경우다.

## 20. ROC Curve와 AUC

ROC Curve는 threshold를 바꿔가며 분류 성능을 보는 그래프다.

AUC는 ROC Curve 아래 면적이다.

```text
AUC가 1에 가까울수록 좋다.
AUC가 0.5에 가까우면 무작위 예측 수준이다.
```

### 함수 사용법

```python
roc_curve(y_true, y_score)
roc_auc_score(y_true, y_score)
```

- `y_score`: class 예측값이 아니라 확률값 또는 점수다.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val_np, val_prob.numpy().ravel())
auc_value = roc_auc_score(y_val_np, val_prob.numpy().ravel())

print("AUC:", auc_value)
print("threshold 개수:", len(thresholds))

In [ ]:
plt.plot(fpr, tpr, label=f"AUC = {auc_value:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

그래프 해석:

- 대각선은 무작위 예측 기준선이다.
- ROC 곡선이 왼쪽 위에 가까울수록 좋다.
- AUC는 threshold 하나에 묶이지 않고 전체적인 분리 능력을 보여준다.

## 21. 사기 거래 탐지 데이터 만들기

사기 거래 탐지는 대표적인 클래스 불균형 문제다.

```text
정상 거래: 대부분
사기 거래: 매우 적음
```

여기서는 정상 95%, 사기 5% 비율의 합성 데이터를 만든다.

### 함수 사용법: `make_classification()`

```python
make_classification(n_samples=5000, weights=[0.95, 0.05])
```

- `weights`: class 비율을 정한다.
- `[0.95, 0.05]`는 class 0이 95%, class 1이 5%라는 뜻이다.

In [ ]:
X_fraud, y_fraud = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_clusters_per_class=2,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=42
)

unique, counts = np.unique(y_fraud, return_counts=True)

print("classes:", unique)
print("counts:", counts)
print("fraud ratio:", counts[1] / counts.sum())

> 기억할 점:  
> 이런 데이터에서 모델이 전부 정상이라고 예측해도 Accuracy가 높게 나올 수 있다.  
> 그래서 Recall, Precision, F1, Confusion Matrix가 중요하다.

## 22. 계층적 샘플링과 표준화

불균형 데이터는 train/test 분할 때 class 비율이 유지되도록 `stratify`를 사용한다.

그리고 입력 feature는 표준화한다.

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud,
    y_fraud,
    test_size=0.2,
    random_state=42,
    stratify=y_fraud
)

scaler_f = StandardScaler()

X_train_f_scaled = scaler_f.fit_transform(X_train_f)
X_test_f_scaled = scaler_f.transform(X_test_f)

print("train fraud ratio:", y_train_f.mean())
print("test fraud ratio:", y_test_f.mean())
print("scaled mean:", X_train_f_scaled.mean().round(4))
print("scaled std:", X_train_f_scaled.std().round(4))

### 함수 사용법: `fit_transform()`과 `transform()`

```python
scaler.fit_transform(X_train)
scaler.transform(X_test)
```

- train에는 `fit_transform`을 사용한다.
- test에는 train에서 배운 기준으로 `transform`만 사용한다.
- test에 `fit_transform`을 쓰면 데이터 누수 위험이 있다.

## 23. 사기 탐지 Tensor와 모델

불균형 이진 분류 모델을 만든다.

모델 출력은 logit 하나다.

```text
Linear → ReLU → Linear → ReLU → Linear(1)
```

In [ ]:
X_train_f_t = torch.FloatTensor(X_train_f_scaled)
X_test_f_t = torch.FloatTensor(X_test_f_scaled)

y_train_f_t = torch.FloatTensor(y_train_f).view(-1, 1)
y_test_f_t = torch.FloatTensor(y_test_f).view(-1, 1)

class FraudDetectionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

fraud_model = FraudDetectionModel(input_dim=20)

print(fraud_model)

## 24. pos_weight 적용하기

양성 class인 사기 거래가 적으므로 `pos_weight`를 사용한다.

### 함수 사용법

```python
pos_weight = torch.tensor([normal_count / fraud_count])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
```

- class 1을 틀렸을 때 더 큰 패널티를 준다.
- 불균형 문제에서 Recall 개선을 기대할 수 있다.

In [ ]:
normal_count = (y_train_f == 0).sum()
fraud_count = (y_train_f == 1).sum()

pos_weight = torch.tensor([normal_count / fraud_count], dtype=torch.float32)

criterion_fraud = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer_fraud = optim.Adam(fraud_model.parameters(), lr=0.001)

print("normal_count:", normal_count)
print("fraud_count:", fraud_count)
print("pos_weight:", pos_weight.item())

## 25. 사기 탐지 모델 학습

학습하면서 train loss와 test F1을 기록한다.

In [ ]:
fraud_history = []

for epoch in range(101):
    fraud_model.train()
    optimizer_fraud.zero_grad()

    logits = fraud_model(X_train_f_t)
    loss = criterion_fraud(logits, y_train_f_t)

    loss.backward()
    optimizer_fraud.step()

    if epoch % 10 == 0:
        fraud_model.eval()
        with torch.no_grad():
            test_logits = fraud_model(X_test_f_t)
            test_probs = torch.sigmoid(test_logits)
            test_pred = (test_probs >= 0.5).float().numpy().ravel()

        f1_tmp = f1_score(y_test_f, test_pred, zero_division=0)
        recall_tmp = recall_score(y_test_f, test_pred, zero_division=0)

        fraud_history.append([epoch, loss.item(), f1_tmp, recall_tmp])

fraud_history = np.array(fraud_history)

print("final train loss:", fraud_history[-1, 1])
print("final F1:", fraud_history[-1, 2])
print("final Recall:", fraud_history[-1, 3])

In [ ]:
plt.plot(fraud_history[:, 0], fraud_history[:, 1], label="train loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Fraud Detection Training Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(fraud_history[:, 0], fraud_history[:, 2], label="F1")
plt.plot(fraud_history[:, 0], fraud_history[:, 3], label="Recall")
plt.xlabel("epoch")
plt.ylabel("score")
plt.title("Fraud Detection F1 and Recall")
plt.legend()
plt.show()

그래프 해석:

- Loss가 내려가면 학습이 진행되는 것이다.
- 불균형 데이터에서는 F1과 Recall을 같이 봐야 한다.
- Recall이 높으면 사기 거래를 더 많이 찾아낸다는 뜻이다.

## 26. 사기 탐지 상세 평가

Confusion Matrix, Classification Report, ROC-AUC를 확인한다.

In [ ]:
fraud_model.eval()

with torch.no_grad():
    test_logits = fraud_model(X_test_f_t)
    test_probs = torch.sigmoid(test_logits).numpy().ravel()
    test_pred = (test_probs >= 0.5).astype(int)

print("[Classification Report]")
print(classification_report(y_test_f, test_pred, target_names=["Normal", "Fraud"], zero_division=0))

cm_fraud = confusion_matrix(y_test_f, test_pred)
auc_fraud = roc_auc_score(y_test_f, test_probs)

print("[Confusion Matrix]")
print(cm_fraud)
print("AUC:", auc_fraud)

In [ ]:
plt.imshow(cm_fraud)
plt.title("Fraud Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm_fraud):
    plt.text(j, i, str(value), ha="center", va="center")

plt.colorbar()
plt.show()

In [ ]:
fpr_f, tpr_f, _ = roc_curve(y_test_f, test_probs)

plt.plot(fpr_f, tpr_f, label=f"AUC={auc_fraud:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Fraud ROC Curve")
plt.legend()
plt.show()

## 27. 당뇨병 예측 실무형 구조

당뇨병 예측 예제는 단순 학습 코드보다 실무 구조에 가깝다.

핵심 구성은 다음이다.

```text
Config
→ DataPreprocessor
→ DiabetesClassifier
→ EarlyStopping
→ Trainer
→ Evaluate
```

> 필기 포인트:  
> 코드가 길어지는 이유는 모델 때문만이 아니다.  
> 설정 관리, 데이터 전처리, 학습 기록, 조기 종료, 평가가 들어가면 실무형 코드가 된다.

In [ ]:
class Config:
    def __init__(self):
        self.test_size = 0.2
        self.val_size = 0.2
        self.random_state = 42
        self.input_dim = 10
        self.hidden_dims = [64, 32, 16]
        self.dropout_rate = 0.3
        self.batch_size = 32
        self.num_epochs = 80
        self.learning_rate = 0.001
        self.weight_decay = 0.0001
        self.patience = 10
        self.min_delta = 0.001
        self.scheduler_step_size = 20
        self.scheduler_gamma = 0.5

config = Config()

print("Config 준비 완료")
print("batch_size:", config.batch_size)
print("learning_rate:", config.learning_rate)

### 클래스 사용법: `Config`

```python
config = Config()
```

- 실험 설정을 한곳에 모아둔다.
- batch size, learning rate, epoch 같은 값을 쉽게 바꿀 수 있다.

## 28. DataPreprocessor 클래스

데이터 전처리를 클래스로 묶는다.

### 주요 역할

```text
1. 데이터 불러오기
2. 회귀 target을 이진 target으로 변환
3. train/validation/test 분할
4. StandardScaler 적용
5. DataLoader 생성
```

In [ ]:
class DataPreprocessor:
    def __init__(self, config):
        self.config = config
        self.scaler = StandardScaler()

    def load_and_prepare_data(self):
        diabetes = load_diabetes()

        X = diabetes.data
        y_regression = diabetes.target

        median = np.median(y_regression)
        y = (y_regression > median).astype(int)

        X_train_val, X_test, y_train_val, y_test = train_test_split(
            X,
            y,
            test_size=self.config.test_size,
            stratify=y,
            random_state=self.config.random_state
        )

        X_train, X_val, y_train, y_val = train_test_split(
            X_train_val,
            y_train_val,
            test_size=self.config.val_size,
            stratify=y_train_val,
            random_state=self.config.random_state
        )

        X_train = self.scaler.fit_transform(X_train)
        X_val = self.scaler.transform(X_val)
        X_test = self.scaler.transform(X_test)

        return X_train, X_val, X_test, y_train, y_val, y_test

    def create_dataloaders(self, X_train, X_val, X_test, y_train, y_val, y_test):
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train),
            torch.FloatTensor(y_train).view(-1, 1)
        )

        val_dataset = TensorDataset(
            torch.FloatTensor(X_val),
            torch.FloatTensor(y_val).view(-1, 1)
        )

        test_dataset = TensorDataset(
            torch.FloatTensor(X_test),
            torch.FloatTensor(y_test).view(-1, 1)
        )

        train_loader = DataLoader(train_dataset, batch_size=self.config.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=self.config.batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=self.config.batch_size, shuffle=False)

        return train_loader, val_loader, test_loader

preprocessor = DataPreprocessor(config)
X_train_d, X_val_d, X_test_d, y_train_d, y_val_d, y_test_d = preprocessor.load_and_prepare_data()
train_loader_d, val_loader_d, test_loader_d = preprocessor.create_dataloaders(
    X_train_d, X_val_d, X_test_d, y_train_d, y_val_d, y_test_d
)

print("Train:", X_train_d.shape)
print("Val:", X_val_d.shape)
print("Test:", X_test_d.shape)

> 주의:  
> scaler는 train에만 `fit_transform`을 적용한다.  
> validation/test에는 train 기준으로 `transform`만 적용한다.

## 29. DiabetesClassifier 모델

실무형 모델에는 BatchNorm과 Dropout을 함께 사용한다.

구조는 다음이다.

```text
Linear → BatchNorm → ReLU → Dropout
```

이 블록을 여러 번 반복한 뒤 마지막에 `Linear(..., 1)`로 logit을 출력한다.

In [ ]:
class DiabetesClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout_rate=0.3):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)
        self._initialize_weights()

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, x):
        return self.network(x)

diabetes_model = DiabetesClassifier(
    config.input_dim,
    config.hidden_dims,
    config.dropout_rate
).to(device)

print(diabetes_model)

### 함수 사용법 정리

```python
nn.BatchNorm1d(hidden_dim)
```

- hidden feature의 분포를 안정화한다.

```python
nn.Dropout(dropout_rate)
```

- 일부 뉴런 출력을 랜덤하게 꺼서 과적합을 줄인다.

```python
nn.init.kaiming_normal_()
```

- ReLU 계열 활성화 함수와 잘 맞는 He 초기화다.

## 30. EarlyStopping 클래스

EarlyStopping은 validation loss가 더 이상 좋아지지 않으면 학습을 멈춘다.

### 클래스 사용법

```python
early_stopping = EarlyStopping(patience=10, min_delta=0.001)
early_stopping(val_loss, model)
```

- `patience`: 몇 번까지 개선을 기다릴지 정한다.
- `min_delta`: 개선으로 인정할 최소 변화량이다.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

early_stopping_demo = EarlyStopping(config.patience, config.min_delta)

print("EarlyStopping 준비 완료")

## 31. Trainer 클래스

Trainer는 학습과 검증 과정을 한 클래스에 묶는다.

핵심 메서드는 다음이다.

```text
train_epoch()
validate()
train()
```

- `train_epoch`: 한 epoch 동안 train_loader로 학습한다.
- `validate`: validation_loader로 성능을 측정한다.
- `train`: 여러 epoch를 반복하고 history를 저장한다.

In [ ]:
class Trainer:
    def __init__(self, model, config):
        self.model = model
        self.config = config

        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )

        self.scheduler = optim.lr_scheduler.StepLR(
            self.optimizer,
            step_size=config.scheduler_step_size,
            gamma=config.scheduler_gamma
        )

        self.early_stopping = EarlyStopping(config.patience, config.min_delta)

        self.history = {
            "train_loss": [],
            "val_loss": [],
            "train_acc": [],
            "val_acc": [],
            "learning_rate": []
        }

    def train_epoch(self, train_loader):
        self.model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            self.optimizer.zero_grad()

            logits = self.model(X_batch)
            loss = self.criterion(logits, y_batch)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            pred = (probs >= 0.5).float()

            correct += (pred == y_batch).sum().item()
            total += y_batch.size(0)

        return total_loss / len(train_loader), correct / total

    def validate(self, val_loader):
        self.model.eval()

        total_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                logits = self.model(X_batch)
                loss = self.criterion(logits, y_batch)

                total_loss += loss.item()

                probs = torch.sigmoid(logits)
                pred = (probs >= 0.5).float()

                correct += (pred == y_batch).sum().item()
                total += y_batch.size(0)

        return total_loss / len(val_loader), correct / total

    def train(self, train_loader, val_loader):
        for epoch in range(self.config.num_epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.validate(val_loader)

            self.scheduler.step()

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_acc"].append(train_acc)
            self.history["val_acc"].append(val_acc)
            self.history["learning_rate"].append(self.optimizer.param_groups[0]["lr"])

            self.early_stopping(val_loss, self.model)

            if self.early_stopping.early_stop:
                break

        if self.early_stopping.best_model_state is not None:
            self.model.load_state_dict(self.early_stopping.best_model_state)

        return self.history

trainer = Trainer(diabetes_model, config)

history_diabetes = trainer.train(train_loader_d, val_loader_d)

print("학습 epoch 수:", len(history_diabetes["train_loss"]))
print("최종 val acc:", history_diabetes["val_acc"][-1])

> 필기 포인트:  
> 실무에서는 학습 루프를 함수나 클래스로 묶어서 재사용한다.  
> 그래야 실험 설정을 바꾸고 비교하기 편하다.

## 32. 당뇨병 예측 학습 결과 시각화

학습 결과는 loss, accuracy, learning rate를 각각 확인한다.

In [ ]:
plt.plot(history_diabetes["train_loss"], label="train loss")
plt.plot(history_diabetes["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Diabetes Loss Curve")
plt.legend()
plt.show()

In [ ]:
plt.plot(history_diabetes["train_acc"], label="train acc")
plt.plot(history_diabetes["val_acc"], label="val acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Diabetes Accuracy Curve")
plt.legend()
plt.show()

In [ ]:
plt.plot(history_diabetes["learning_rate"], label="learning rate")
plt.xlabel("epoch")
plt.ylabel("learning rate")
plt.title("Learning Rate Schedule")
plt.legend()
plt.show()

그래프 해석:

- train loss와 val loss가 함께 내려가면 학습이 안정적이다.
- train accuracy만 높고 val accuracy가 낮으면 과적합을 의심한다.
- learning rate curve는 scheduler가 학습률을 어떻게 바꾸는지 보여준다.

## 33. 당뇨병 예측 모델 평가

테스트 데이터에서 classification report, confusion matrix, AUC를 확인한다.

In [ ]:
def evaluate_binary_model(model, data_loader):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch = X_batch.to(device)

            logits = model(X_batch)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()

            all_probs.extend(probs)
            all_labels.extend(y_batch.numpy().ravel())

    all_labels = np.array(all_labels).astype(int)
    all_probs = np.array(all_probs)
    all_preds = (all_probs >= 0.5).astype(int)

    return all_labels, all_probs, all_preds

diabetes_labels, diabetes_probs, diabetes_preds = evaluate_binary_model(diabetes_model, test_loader_d)

print("[Classification Report]")
print(classification_report(diabetes_labels, diabetes_preds, target_names=["Low Risk", "High Risk"], zero_division=0))

cm_diabetes = confusion_matrix(diabetes_labels, diabetes_preds)
auc_diabetes = roc_auc_score(diabetes_labels, diabetes_probs)

print("[Confusion Matrix]")
print(cm_diabetes)
print("AUC:", auc_diabetes)

In [ ]:
precision, recall, _ = precision_recall_curve(diabetes_labels, diabetes_probs)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Diabetes Precision-Recall Curve")
plt.show()

그래프 해석:

- Precision-Recall Curve는 양성 class 탐지 품질을 볼 때 유용하다.
- 의료 예측처럼 양성 class를 놓치면 위험한 문제에서는 Recall 관점이 중요할 수 있다.

## 34. 고객 이탈 예측 데이터 만들기

고객 이탈 예측은 실무형 이진 분류 문제다.

```text
입력: 고객 이용 기간, 월요금, 계약 형태, 결제 방식 등
출력: 이탈 여부 0/1
```

수치형과 범주형 feature가 섞여 있으므로 전처리 파이프라인이 중요하다.

In [ ]:
np.random.seed(42)

N = 5000

tenure = np.random.randint(0, 72, size=N)

monthly_charges = np.round(np.random.normal(60, 15, size=N), 2)
monthly_charges = np.clip(monthly_charges, 5, 200)

total_charges = np.round(monthly_charges * (tenure + np.random.normal(0.0, 1.0, size=N)), 2)

support_calls = np.random.poisson(lam=1.8, size=N)

contract_type = np.random.choice(
    ["month-to-month", "one-year", "two-year"],
    size=N,
    p=[0.6, 0.25, 0.15]
)

has_internet = np.random.choice(["yes", "no"], size=N, p=[0.8, 0.2])

has_giga = np.where(
    (has_internet == "yes") & (np.random.rand(N) < 0.3),
    "yes",
    "no"
)

add_on = np.random.choice(
    ["none", "security", "streaming", "both"],
    size=N,
    p=[0.4, 0.25, 0.25, 0.10]
)

payment_method = np.random.choice(
    ["credit_card", "bank_transfer", "e_check", "cash"],
    size=N,
    p=[0.35, 0.25, 0.30, 0.10]
)

logit = (
    -2.0
    + 0.03 * (70 - tenure)
    + 0.015 * (monthly_charges - 60)
    + 0.25 * support_calls
    + np.where(contract_type == "month-to-month", 0.8, 0.0)
    + np.where(payment_method == "e_check", 0.5, 0.0)
    + np.where(has_internet == "no", 0.3, 0.0)
    + np.where((has_internet == "yes") & (has_giga == "no") & (monthly_charges > 80), 0.25, 0.0)
)

prob = 1 / (1 + np.exp(-logit))
churn = np.random.binomial(1, prob, size=N)

df = pd.DataFrame({
    "tenure": tenure,
    "monthly_charges": monthly_charges,
    "total_charges": total_charges,
    "support_calls": support_calls,
    "contract_type": contract_type,
    "has_internet": has_internet,
    "has_giga": has_giga,
    "add_on": add_on,
    "payment_method": payment_method,
    "churn": churn
})

print(df.head())
print("churn ratio:", df["churn"].mean())

## 35. 수치형/범주형 전처리 파이프라인

고객 이탈 데이터는 수치형과 범주형이 섞여 있다.

수치형은 표준화하고, 범주형은 One-Hot Encoding을 적용한다.

### 함수 사용법

```python
ColumnTransformer(...)
Pipeline(...)
OneHotEncoder(handle_unknown="ignore")
```

- `ColumnTransformer`: 열 종류별로 다른 전처리를 적용한다.
- `Pipeline`: 전처리와 모델을 하나로 묶는다.
- `handle_unknown="ignore"`: 테스트에 새로운 범주가 나와도 에러를 막는다.

In [ ]:
X_churn = df.drop(columns=["churn"])
y_churn = df["churn"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_churn,
    y_churn,
    test_size=0.2,
    random_state=42,
    stratify=y_churn
)

numeric_features = ["tenure", "monthly_charges", "total_charges", "support_calls"]
categorical_features = ["contract_type", "has_internet", "has_giga", "add_on", "payment_method"]

numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("전처리 파이프라인 준비 완료")

## 36. LogisticRegression 파이프라인

고객 이탈 예측에서는 scikit-learn의 LogisticRegression을 사용한다.

### 함수 사용법

```python
LogisticRegression(class_weight="balanced")
```

- `class_weight="balanced"`는 class 비율을 고려해 가중치를 조정한다.
- 이탈 class가 상대적으로 적을 때 도움이 될 수 있다.

In [ ]:
clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", clf)
])

pipe.fit(X_train_c, y_train_c)

y_proba_c = pipe.predict_proba(X_test_c)[:, 1]
y_pred_c = (y_proba_c >= 0.5).astype(int)

print("pipeline 학습 완료")

### 함수 사용법: `predict_proba()`

```python
pipe.predict_proba(X_test)[:, 1]
```

- 각 class에 대한 예측 확률을 반환한다.
- `[:, 1]`은 class 1, 즉 이탈 확률만 가져온다.

## 37. 고객 이탈 기본 평가

기본 threshold 0.5에서 평가한다.

In [ ]:
acc_c = accuracy_score(y_test_c, y_pred_c)
prec_c = precision_score(y_test_c, y_pred_c, zero_division=0)
rec_c = recall_score(y_test_c, y_pred_c, zero_division=0)
f1_c = f1_score(y_test_c, y_pred_c, zero_division=0)
auc_c = roc_auc_score(y_test_c, y_proba_c)

print("Accuracy:", acc_c)
print("Precision:", prec_c)
print("Recall:", rec_c)
print("F1:", f1_c)
print("ROC-AUC:", auc_c)

print("[Classification Report]")
print(classification_report(y_test_c, y_pred_c, zero_division=0))

In [ ]:
cm_c = confusion_matrix(y_test_c, y_pred_c)

plt.imshow(cm_c)
plt.title("Churn Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

for (i, j), value in np.ndenumerate(cm_c):
    plt.text(j, i, str(value), ha="center", va="center")

plt.colorbar()
plt.show()

In [ ]:
fpr_c, tpr_c, _ = roc_curve(y_test_c, y_proba_c)

plt.plot(fpr_c, tpr_c, label=f"AUC={auc_c:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Churn ROC Curve")
plt.legend()
plt.show()

## 38. 임계값 Threshold 최적화

기본 threshold 0.5가 항상 최선은 아니다.

비즈니스 목적에 따라 threshold를 조정할 수 있다.

```text
Recall을 높이고 싶음 → threshold 낮춤
Precision을 높이고 싶음 → threshold 높임
F1을 최대로 하고 싶음 → 여러 threshold를 비교
```

In [ ]:
thresholds = np.linspace(0.1, 0.9, 81)

best_thr = 0.5
best_f1 = f1_c

threshold_records = []

for thr in thresholds:
    y_hat = (y_proba_c >= thr).astype(int)
    f1_tmp = f1_score(y_test_c, y_hat, zero_division=0)
    rec_tmp = recall_score(y_test_c, y_hat, zero_division=0)
    prec_tmp = precision_score(y_test_c, y_hat, zero_division=0)

    threshold_records.append([thr, prec_tmp, rec_tmp, f1_tmp])

    if f1_tmp > best_f1:
        best_f1 = f1_tmp
        best_thr = thr

threshold_records = np.array(threshold_records)

print("Best Threshold:", best_thr)
print("Best F1:", best_f1)

In [ ]:
plt.plot(threshold_records[:, 0], threshold_records[:, 1], label="Precision")
plt.plot(threshold_records[:, 0], threshold_records[:, 2], label="Recall")
plt.plot(threshold_records[:, 0], threshold_records[:, 3], label="F1")
plt.axvline(best_thr, linestyle="--", label="best threshold")
plt.xlabel("threshold")
plt.ylabel("score")
plt.title("Threshold Optimization")
plt.legend()
plt.show()

그래프 해석:

- threshold가 낮아지면 1로 예측하는 샘플이 많아져 recall이 올라가는 경향이 있다.
- threshold가 높아지면 더 확실한 경우만 1로 예측해서 precision이 올라갈 수 있다.
- F1 기준 최적 threshold는 precision과 recall의 균형점이다.

## 39. 최적 임계값으로 재평가

찾은 threshold를 적용해서 다시 평가한다.

In [ ]:
y_pred_opt_c = (y_proba_c >= best_thr).astype(int)

print("Accuracy:", accuracy_score(y_test_c, y_pred_opt_c))
print("Precision:", precision_score(y_test_c, y_pred_opt_c, zero_division=0))
print("Recall:", recall_score(y_test_c, y_pred_opt_c, zero_division=0))
print("F1:", f1_score(y_test_c, y_pred_opt_c, zero_division=0))

print("[Optimized Classification Report]")
print(classification_report(y_test_c, y_pred_opt_c, zero_division=0))

> 실무 포인트:  
> threshold는 단순히 0.5로 고정하는 것이 아니라, 문제의 비용 구조에 따라 조정할 수 있다.  
> 예를 들어 사기 거래를 놓치는 비용이 크면 recall을 높이는 방향으로 threshold를 조정할 수 있다.

## 40. 고객 이탈 특성 중요도 확인

로지스틱 회귀는 계수로 어느 feature가 이탈 확률에 영향을 주는지 볼 수 있다.

범주형 feature는 One-Hot Encoding 이후 여러 열로 나뉜다.

In [ ]:
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(categorical_features)

feature_names = numeric_features + list(cat_names)

coef = pipe.named_steps["model"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coef
}).sort_values("coef", ascending=False)

print(coef_df.head(10))
print("\n가장 음수 방향 feature")
print(coef_df.tail(10))

In [ ]:
top_coef = coef_df.reindex(coef_df["coef"].abs().sort_values(ascending=False).index).head(10)

plt.bar(range(len(top_coef)), top_coef["coef"].values)
plt.xticks(range(len(top_coef)), top_coef["feature"].values, rotation=45, ha="right")
plt.ylabel("coefficient")
plt.title("Top Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

그래프 해석:

- 양수 계수는 이탈 확률을 높이는 방향으로 작용한다.
- 음수 계수는 이탈 확률을 낮추는 방향으로 작용한다.
- 단, 계수 해석은 전처리와 feature scale을 함께 고려해야 한다.

## 41. Permutation Importance

Permutation Importance는 특정 feature를 섞었을 때 모델 성능이 얼마나 떨어지는지 보는 방법이다.

### 함수 사용법

```python
permutation_importance(model, X, y, scoring="f1")
```

- feature를 하나씩 섞어서 성능 변화를 본다.
- 성능이 많이 떨어지면 중요한 feature로 볼 수 있다.

In [ ]:
perm = permutation_importance(
    pipe,
    X_test_c,
    y_test_c,
    n_repeats=5,
    random_state=42,
    scoring="f1"
)

sorted_idx = perm.importances_mean.argsort()[::-1][:10]

plt.bar(range(len(sorted_idx)), perm.importances_mean[sorted_idx])
plt.xticks(range(len(sorted_idx)), np.array(X_test_c.columns)[sorted_idx], rotation=45, ha="right")
plt.ylabel("importance")
plt.title("Permutation Importance Top 10")
plt.tight_layout()
plt.show()

> 기억할 점:  
> 계수 중요도는 모델 내부 관점이고, permutation importance는 예측 성능 변화 관점이다.  
> 둘을 함께 보면 해석이 더 안정적이다.

## 42. 모델 저장과 불러오기

실무에서는 학습한 파이프라인을 파일로 저장한다.

### 함수 사용법

```python
joblib.dump(pipe, "model.joblib")
loaded = joblib.load("model.joblib")
```

- `dump`: 모델을 파일로 저장한다.
- `load`: 저장된 모델을 다시 불러온다.
- 전처리와 모델을 Pipeline으로 묶어두면 같이 저장할 수 있다.

In [ ]:
model_path = "/mnt/data/churn_model_pipeline_day8.joblib"

joblib.dump(pipe, model_path)

loaded = joblib.load(model_path)

test_example = X_test_c.iloc[[0]]
pred_proba_example = loaded.predict_proba(test_example)[:, 1][0]

print("저장 경로:", model_path)
print("샘플 1건 이탈 확률:", pred_proba_example)

## 43. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `logit` | Sigmoid 전 raw score | `model(x)` 출력 |
| `prob` | 0~1 확률 | `torch.sigmoid(logit)` |
| `threshold` | class 결정 기준 | 보통 0.5 |
| `Sigmoid` | 0~1 변환 함수 | `torch.sigmoid(x)` |
| `BCE` | Binary Cross Entropy | 이진 분류 손실 |
| `BCELoss` | 확률값용 BCE | `nn.BCELoss()` |
| `BCEWithLogitsLoss` | logit용 BCE | `nn.BCEWithLogitsLoss()` |
| `Accuracy` | 전체 중 맞춘 비율 | `accuracy_score()` |
| `Precision` | 1 예측 중 실제 1 비율 | `precision_score()` |
| `Recall` | 실제 1 중 찾은 비율 | `recall_score()` |
| `F1` | Precision과 Recall 균형 | `f1_score()` |
| `Confusion Matrix` | 예측/정답 2×2 표 | `confusion_matrix()` |
| `ROC-AUC` | threshold 전체 분리 성능 | `roc_auc_score()` |
| `stratify` | class 비율 유지 분할 | `train_test_split(..., stratify=y)` |
| `pos_weight` | 양성 class 가중치 | `BCEWithLogitsLoss(pos_weight=...)` |
| `DataLoader` | mini-batch 생성 | `DataLoader(dataset, batch_size=...)` |
| `EarlyStopping` | 조기 종료 | val loss 개선 없으면 중단 |
| `Pipeline` | 전처리+모델 묶음 | scikit-learn 실무 구조 |
| `OneHotEncoder` | 범주형 인코딩 | category를 0/1 열로 변환 |
| `Permutation Importance` | 특성 중요도 | feature 섞은 뒤 성능 하락 확인 |

## 44. 시험용 요약

```text
이진 분류 = 데이터를 0 또는 1 두 그룹 중 하나로 나누는 문제
```

꼭 기억할 것:

- 회귀는 연속값 예측이고, 이진 분류는 0/1 class 예측이다.
- 이진 분류 모델은 보통 class 1일 확률을 예측한다.
- Sigmoid는 어떤 값을 0과 1 사이로 바꾼다.
- Sigmoid 입력 0의 출력은 0.5다.
- 확률이 0.5 이상이면 class 1로 예측하는 것이 기본이다.
- BCELoss는 Sigmoid를 지난 확률값을 입력으로 받는다.
- BCEWithLogitsLoss는 Sigmoid 전 logit을 입력으로 받는다.
- BCEWithLogitsLoss는 수치적으로 더 안정적이다.
- BCEWithLogitsLoss를 쓰면 모델 마지막에 Sigmoid를 붙이지 않는다.
- Accuracy는 전체 중 맞힌 비율이다.
- 불균형 데이터에서는 Accuracy만 보면 위험하다.
- Precision은 1이라고 예측한 것 중 실제 1의 비율이다.
- Recall은 실제 1 중 모델이 찾아낸 비율이다.
- F1은 Precision과 Recall의 균형이다.
- Confusion Matrix는 TN, FP, FN, TP를 보여준다.
- ROC-AUC는 threshold 전체에서의 분리 성능이다.
- stratify는 train/test class 비율을 유지하는 데 필요하다.
- StandardScaler는 train에는 fit_transform, test에는 transform만 사용한다.
- pos_weight는 소수 양성 class를 더 중요하게 학습시키는 방법이다.
- EarlyStopping은 validation loss가 좋아지지 않으면 학습을 멈춘다.
- Threshold는 0.5로 고정하지 않고 목적에 따라 조정할 수 있다.
- 고객 이탈 예측처럼 수치형과 범주형이 섞이면 ColumnTransformer와 Pipeline이 유용하다.
- 모델 저장은 joblib.dump, 불러오기는 joblib.load를 사용한다.